# Bayesian regularization scan with Gaussian reco toys

Forward-fold a deliberately distorted Gen prior with the full-MC detector model, give the resulting inclusive-reco pseudo-data the statistical precision of trigger-stitched data, and select a Bayesian iteration count from paired pseudo-experiments. Every toy uses the same factorized purity–matched-migration–efficiency procedure as the closure notebooks. The result is specific to this response, prior deformation, binning, and uncertainty model.

## Statistical construction

The target truth $t$ is a deliberately distorted prior whose $p_T^{ave}$ block normalizations are adjusted until its forward-folded reco yields match the selected data triggers. The detector expectation is

$$\mu_i=\sum_j A_{ij}t_j+f_i,$$

where $A$ includes matched migration and inefficiency and $f_i$ is restored from the MC fake-to-matched-reco ratio. Data provide only the absolute uncertainties $\sigma_i$. Toy $k$ is drawn independently by bin as $m_i^{(k)}\sim\mathcal N(\mu_i,\sigma_i)$. The same toys are reused at every Bayesian iteration, so differences between iteration metrics are paired rather than being obscured by different random ensembles.

Every toy follows

$$m^{all,(k)} \xrightarrow{\times P_i} m^{matched,(k)} \xrightarrow{U_n[M]} \hat t^{matched,(k)} \xrightarrow{/\epsilon_j} \hat t^{all,(k)}_n.$$

For iteration $n$ and truth bin $j$,

$$\bar t_{j,n}=\frac1K\sum_k\hat t_{j,n}^{(k)}, \quad b_{j,n}=\bar t_{j,n}-t_j,$$
$$V_{j,n}=\frac1{K-1}\sum_k(\hat t_{j,n}^{(k)}-\bar t_{j,n})^2, \quad MSE_{j,n}=b_{j,n}^2+V_{j,n}.$$

The absolute summary averages these actual per-bin quantities, $N^{-1}\sum_j q_{j,n}$; consequently larger absolute discrepancies contribute more. The relative summary $N_{valid}^{-1}\sum_j q_{j,n}/t_j^2$ is a separate fractional diagnostic. `absolute`, `relative`, and `both` selection modes choose the corresponding MSE minimum; `both` records both and uses the absolute result for the single downstream closure.

<!-- detailed-workflow-guide -->

### Detailed mathematical workflow

The target truth $t$ is forward-folded to $\mu_i=\sum_jA_{ij}t_j+f_i$, and reco toys are drawn as $m_i^{(k)}\sim\mathcal N(\mu_i,\sigma_i)$. Each toy follows $m^{all}\to Pm^{all}\to U_n[Pm^{all}]\to U_n/\epsilon$. The same seeded toy ensemble is used for every iteration, giving a paired comparison.

For truth bin $j$, $b_{j,n}=\bar t_{j,n}-t_j$, $V_{j,n}=\sum_k(\hat t_{j,n}^{(k)}-\bar t_{j,n})^2/(K-1)$, and $MSE_{j,n}=b_{j,n}^2+V_{j,n}$. Absolute summaries average $q_{j,n}$ directly; fractional summaries average $q_{j,n}/t_j^2$ over valid bins. Selection may use `absolute`, `relative`, or `both`; `both` records both minima and uses absolute for the single final product. Complete seed-tagged ROOT archives make every toy and metric reproducible without rerunning unfolding.

## 1. Environment

In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import math
import os
import sys

PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / 'CMakeLists.txt').is_file()
                     and (path / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root, load_roounfold
ROOT = load_root(batch=True)
ROOUNFOLD_ROOT, ROOUNFOLD_LIBRARY = load_roounfold(ROOT, project_root=PROJECT_ROOT)
import RooUnfold
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_PTAVE_BINS, STANDARD_DIJET_ETA_CUT_INDEX, TEST_DIJET_PTAVE_BINS,
)
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_data_file, resolve_direction_file,
)
from hist_analysis.python.unfolding import (
    FlattenedBinning, RegularizationDistributionWriter, UnfoldingInputKeys,
    apply_efficiency_correction, extract_eta_block,
    apply_purity_correction, as_pt_intervals, build_roounfold_response,
    calculate_response_diagnostics, flatten_pt_eta_projections,
    flatten_sparse_response, forward_fold_truth, load_unfolding_inputs,
    copy_histogram_errors, histogram_fingerprint,
    load_regularization_scan_cache, load_toy_cache, make_gaussian_toys,
    match_truth_to_reco_pt_yields, prepare_factorized_corrections,
    project_eta_by_pt,
    regularization_distribution_file_matches,
    scan_bayes_regularization, unfold_bayes,
    write_regularization_scan_cache, write_toy_cache, write_unfolding_output,
)
from hist_analysis.python.unfolding_plots import (
    draw_regularization_metrics, draw_regularization_metrics_by_pt,
    draw_toy_input_diagnostic,
)
from hist_analysis.python.plotting import draw_closure

## 2. Configuration

For a final result use `standard` bins. The wider `test` bins are intended only for a quick code check; their trigger choices are shown explicitly below.

In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
GENERATOR = 'embedding'
DIRECTION = 'Pbgoing'
FILE_STEM = 'jetId'
DATA_SELECTION = 'jetId'

ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.3, 2.4, 3.0)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
PTAVE_BIN_SET = 'test'          # test or standard
PTAVE_BIN_SETS = {'test': TEST_DIJET_PTAVE_BINS, 'standard': DIJET_PTAVE_BINS}

# Statistical ensemble and Bayesian scan. Reusing a seed reproduces the toys.
N_TOYS = 1000
MAX_ITERATIONS = 20
RANDOM_SEED = 12345
# 'absolute': populated bins carry their natural larger weight.
# 'relative': optimize a typical fractional bin discrepancy.
# 'both': report both minima and use absolute for the singular final output.
REGULARIZATION_SELECTION_MODE = 'absolute'
RESPONSE_SAMPLE = 'full'
TOY_DISTRIBUTION = 'gaussian'
STOP_ON_NEGATIVE_TOY_BIN = True
# RooUnfold is invariant under this common response-statistics scale. It
# suppresses response-training statistical errors in this toy-only study.
RESPONSE_SCALE = 1.0e12
YIELD_MATCH_TOLERANCE = 1.0e-8
YIELD_MATCH_MAX_ITERATIONS = 200
YIELD_MATCH_RELAXATION = 0.5

PLOT_PER_PT_METRICS = False  # optional slices; not used to select iterations
LOG_REGULARIZATION_METRICS_Y = True
DRAW_GRID = True
SAVE_PNG = False
USE_INTERMEDIATE_CACHE = True
STORE_ITERATION_DISTRIBUTIONS = True  # can produce a large ROOT file

STANDARD_TRIGGER_BY_INTERVAL = {
    (50, 60): 'MinimumBias', (60, 70): 'MinimumBias',
    (70, 80): 'MinimumBias', (80, 90): 'Jet60', (90, 100): 'Jet60',
    (100, 110): 'Jet80', (110, 120): 'Jet80',
    (120, 130): 'Jet100', (130, 140): 'Jet100',
    (140, 150): 'Jet100', (150, 160): 'Jet100',
    (160, 180): 'Jet100', (180, 200): 'Jet100',
    (200, 250): 'Jet100', (250, 300): 'Jet100',
    (300, 500): 'Jet100', (500, 1000): 'Jet100',
}
TEST_TRIGGER_BY_INTERVAL = {
    (50, 100): 'MinimumBias', (100, 180): 'Jet80',
    (180, 300): 'Jet100', (300, 500): 'Jet100',
    (500, 1000): 'Jet100',
}
TRIGGER_BY_INTERVAL = (TEST_TRIGGER_BY_INTERVAL if PTAVE_BIN_SET == 'test'
                       else STANDARD_TRIGGER_BY_INTERVAL)

if PTAVE_BIN_SET not in PTAVE_BIN_SETS:
    raise ValueError(f'Unsupported PTAVE_BIN_SET={PTAVE_BIN_SET!r}')
if N_TOYS < 2 or MAX_ITERATIONS < 1:
    raise ValueError('N_TOYS >= 2 and MAX_ITERATIONS >= 1 are required')
if REGULARIZATION_SELECTION_MODE not in {'absolute', 'relative', 'both'}:
    raise ValueError(
        "REGULARIZATION_SELECTION_MODE must be 'absolute', 'relative', or 'both'")
if RESPONSE_SAMPLE != 'full' or TOY_DISTRIBUTION != 'gaussian':
    raise ValueError('This notebook implements the full response and Gaussian toys')
if YIELD_MATCH_TOLERANCE <= 0.0 or YIELD_MATCH_MAX_ITERATIONS < 1:
    raise ValueError('Yield-matching tolerance and iteration limit must be positive')
if not 0.0 < YIELD_MATCH_RELAXATION <= 1.0:
    raise ValueError('YIELD_MATCH_RELAXATION must be in (0, 1]')
if not isinstance(LOG_REGULARIZATION_METRICS_Y, bool):
    raise TypeError('LOG_REGULARIZATION_METRICS_Y must be True or False')
if not isinstance(STORE_ITERATION_DISTRIBUTIONS, bool):
    raise TypeError('STORE_ITERATION_DISTRIBUTIONS must be True or False')

PTAVE_INTERVALS = as_pt_intervals(PTAVE_BIN_SETS[PTAVE_BIN_SET])
configured_intervals = set(PTAVE_INTERVALS)
trigger_intervals = set(as_pt_intervals(TRIGGER_BY_INTERVAL))
missing_trigger_intervals = sorted(configured_intervals - trigger_intervals)
unused_trigger_intervals = sorted(trigger_intervals - configured_intervals)
if missing_trigger_intervals or unused_trigger_intervals:
    raise ValueError(
        'Trigger map does not match configured pT intervals. '
        f'Missing: {missing_trigger_intervals}; unused: {unused_trigger_intervals}'
    )
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
ETA_UNFOLDING_RANGE = (-ETA_CUT, ETA_CUT)
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_UNFOLD2D_REGULARIZATION_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis/output/unfold2D_regularization',
))
OUTPUT_BASE_TAG = (f'{GENERATOR}_{DIRECTION}_{PTAVE_BIN_SET}_eta_{ETA_CUT:g}_'
                   f'{N_TOYS}toys_{MAX_ITERATIONS}iter')
OUTPUT_TAG = f'{OUTPUT_BASE_TAG}_seed_{RANDOM_SEED}'
OUTPUT_ROOT_FILE = OUTPUT_DIR / f'{OUTPUT_TAG}.root'
CACHE_DIR = OUTPUT_DIR / 'intermediate'
TOY_CACHE_BASE_TAG = (f'{GENERATOR}_{DIRECTION}_{PTAVE_BIN_SET}_eta_{ETA_CUT:g}_'
                      f'{N_TOYS}toys')
TOY_CACHE_FILE = CACHE_DIR / f'{TOY_CACHE_BASE_TAG}_toys_seed_{RANDOM_SEED}.root'
SCAN_CACHE_FILE = CACHE_DIR / f'{OUTPUT_BASE_TAG}_scan_seed_{RANDOM_SEED}.root'
ITERATION_DISTRIBUTIONS_FILE = (
    CACHE_DIR / f'{OUTPUT_BASE_TAG}_iteration_distributions_seed_{RANDOM_SEED}.root')

## 3. Load and flatten the full-MC response and distorted prior

In [ ]:
# Cell role: resolve input files and load the named ROOT objects.
# Interpretation: Loaded objects that outlive their file must be cloned and detached from ROOT directories.
# The preceding Markdown gives the equations and physics assumptions for this step.
# Resolve the training sample without embedding machine-specific paths here.
if DIRECTION == 'combined':
    mc_file = resolve_combined_file(BASE_DIR, GENERATOR, FILE_STEM)
else:
    mc_file = resolve_direction_file(BASE_DIR, GENERATOR, DIRECTION, FILE_STEM)

keys = UnfoldingInputKeys(
    truth=f'hGenDijetPtEtaCM_{ETA_CUT_INDEX}',
    measured=f'hRecoDijetPtEtaCMJerDefExtraUnfold_{ETA_CUT_INDEX}',
    response=f'hGenDijetPtEtaCMVsRecoJerDefExtraPtEtaCM_{ETA_CUT_INDEX}',
    miss=f'hGenDijetPtEtaCMMissJerDefExtra_{ETA_CUT_INDEX}',
    fake=f'hRecoDijetPtEtaCMFakeJerDefExtra_{ETA_CUT_INDEX}',
)
inputs = load_unfolding_inputs(mc_file, keys)
prior_2d = load_histogram(str(mc_file), f'hGenDijetPtEtaCMMixedPrior_{ETA_CUT_INDEX}')

# First project every TH2 into one eta histogram per pTave interval. The same
# accepted eta bins are then packed consecutively into each flattened TH1.
projection_options = {'eta_range': ETA_UNFOLDING_RANGE}
truth_by_pt = project_eta_by_pt(
    inputs.truth, PTAVE_INTERVALS, name_prefix='hTrainingTruth',
    **projection_options)
reco_by_pt = project_eta_by_pt(
    inputs.measured, PTAVE_INTERVALS, name_prefix='hTrainingReco',
    **projection_options)
miss_by_pt = project_eta_by_pt(
    inputs.miss, PTAVE_INTERVALS, name_prefix='hTrainingMiss',
    **projection_options)
fake_by_pt = project_eta_by_pt(
    inputs.fake, PTAVE_INTERVALS, name_prefix='hTrainingFake',
    **projection_options)
prior_by_pt = project_eta_by_pt(
    prior_2d, PTAVE_INTERVALS, name_prefix='hMixedPrior',
    **projection_options)

training_truth, layout = flatten_pt_eta_projections(
    truth_by_pt, name='hTrainingTruth', pt_bins=PTAVE_INTERVALS)
training_reco, _ = flatten_pt_eta_projections(reco_by_pt, name='hTrainingReco', layout=layout)
training_miss, _ = flatten_pt_eta_projections(miss_by_pt, name='hTrainingMiss', layout=layout)
training_fake, _ = flatten_pt_eta_projections(fake_by_pt, name='hTrainingFake', layout=layout)
prior, _ = flatten_pt_eta_projections(prior_by_pt, name='hMixedPrior', layout=layout)
response_matrix, _ = flatten_sparse_response(
    inputs.response, PTAVE_INTERVALS, name='hResponseEtaCM', layout=layout,
    eta_range=ETA_UNFOLDING_RANGE)

# Matrix marginals define matched truth/reco. Their differences from the
# inclusive spectra define inefficiency (misses) and impurity (fakes).
diagnostics = calculate_response_diagnostics(
    response_matrix, training_truth, training_reco,
    explicit_miss=training_miss, explicit_fake=training_fake)
# The inclusive response is used only to generate reco-level pseudo-data.
inclusive_response_bundle = build_roounfold_response(
    RooUnfold, training_truth, training_reco, response_matrix,
    diagnostics=diagnostics, scale=RESPONSE_SCALE, require_fakes=True,
    name='regularizationInclusiveResponse')

# These fixed MC corrections define the factorized unfolding applied to every toy.
factorized_inputs = prepare_factorized_corrections(
    training_reco, training_reco, diagnostics.matched_measured,
    training_truth, diagnostics.matched_truth,
    name_prefix='hRegularizationFactorized')
reco_purity = factorized_inputs.purity
truth_efficiency = factorized_inputs.efficiency
matched_response_bundle = build_roounfold_response(
    RooUnfold, diagnostics.matched_truth, diagnostics.matched_measured,
    response_matrix, scale=RESPONSE_SCALE, require_fakes=False,
    name='regularizationMatchedResponse')
print(f'MC input: {mc_file}')
print(f'Global bins: {layout.n_global_bins}')

## 4. Build trigger-stitched data uncertainties

Each configured pT interval uses one explicitly listed trigger. The raw data projection supplies both the block yield and the absolute bin errors.

In [ ]:
# Cell role: resolve input files and load the named ROOT objects.
# Interpretation: Loaded objects that outlive their file must be cloned and detached from ROOT directories.
# The preceding Markdown gives the equations and physics assumptions for this step.
# Cache each trigger histogram in memory because adjacent pT bins can use it.
data_maps = {}
data_by_pt = []
for pt_index, interval in enumerate(PTAVE_INTERVALS):
    trigger = TRIGGER_BY_INTERVAL[interval]
    if trigger not in data_maps:
        data_file = resolve_data_file(
            BASE_DIR / 'exp', trigger, DIRECTION, DATA_SELECTION)
        data_maps[trigger] = load_histogram(str(data_file), 'hRecoDijetPtEtaCM')
    projection = project_eta_by_pt(
        data_maps[trigger], (interval,),
        name_prefix=f'hData{trigger}Pt{pt_index}',
        eta_range=ETA_UNFOLDING_RANGE)[0]
    data_by_pt.append(projection)

data_reco, _ = flatten_pt_eta_projections(
    data_by_pt, name='hTriggerStitchedData', layout=layout)
print('Trigger assignment:')
for interval in PTAVE_INTERVALS:
    print(f'  {interval}: {TRIGGER_BY_INTERVAL[interval]}')

## 5. Match truth exposure, then forward-fold the exact toy truth

Data are used only to set one total yield per pTave block and the absolute bin errors. Because reconstructed pTave migrates between blocks, a reco-block scale factor cannot be copied directly onto the corresponding truth block. Instead, iteratively scale the truth blocks and forward-fold after every update until the folded reco block yields match data. The converged scaled truth is the exact target used in the toy bias calculation. Fakes are transferred with the MC fake/matched-reco ratio, so the generated inclusive reco is algebraically compatible with the purity correction used before unfolding.

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
# Adjust truth-block normalizations through repeated forward folding. This is
# necessary because off-diagonal pT migration couples reco and truth blocks.
yield_match = match_truth_to_reco_pt_yields(
    inclusive_response_bundle, prior, data_reco, diagnostics, layout,
    name_prefix='hMixedPriorDataYield', tolerance=YIELD_MATCH_TOLERANCE,
    max_iterations=YIELD_MATCH_MAX_ITERATIONS,
    relaxation=YIELD_MATCH_RELAXATION)
target_prior = yield_match.truth
target_forward_fold = forward_fold_truth(
    inclusive_response_bundle, target_prior, diagnostics=diagnostics,
    add_fakes=True, fake_normalization='matched_fraction',
    name='hForwardFoldedPriorDataYield')
folded_prior = target_forward_fold.histogram
# Contents come from the detector-folded prior; errors come from real data.
# Thus toys test unfolding regularization at approximately data precision.
toy_expectation = copy_histogram_errors(
    folded_prior, data_reco, name='hRecoToyExpectation')
block_scale_factors = yield_match.factors

print(f'ApplyToTruth included fakes: {target_forward_fold.apply_to_truth_includes_fakes}')
print(f'Yield matching converged in {yield_match.iterations} iterations')
print(f'Maximum reco-block yield residual: {yield_match.maximum_relative_residual:.3g}')
print(f'Block normalization factors: {block_scale_factors}')
print('Gaussian input precision (mu/sigma):')
negative_probabilities = []
for global_bin in range(1, layout.n_global_bins + 1):
    mean = toy_expectation.GetBinContent(global_bin)
    sigma = toy_expectation.GetBinError(global_bin)
    ratio = math.inf if sigma == 0.0 else mean / sigma
    probability = 0.0 if sigma == 0.0 else 0.5 * math.erfc(ratio / math.sqrt(2.0))
    negative_probabilities.append(probability)
    pt_index, eta_bin = layout.indices(global_bin)
    print(f'  bin {global_bin:3d}, pT block {pt_index}, eta bin {eta_bin}: {ratio:.4g}')
expected_negative_draws = N_TOYS * sum(negative_probabilities)
print(f'Expected negative bin draws across all toys: {expected_negative_draws:.3g}')
print(f'Largest single-bin negative probability: {max(negative_probabilities):.3g}')

## 6. Generate one paired Gaussian toy ensemble

For every inclusive-reco toy, the bin content is drawn from N(mu, sigma), while the stored bin error remains the corresponding real-data sigma. A negative draw stops execution instead of silently changing the distribution. Before unfolding, every toy is multiplied bin by bin by the fixed full-MC purity; both its content and error are scaled.

With `USE_INTERMEDIATE_CACHE = True`, generated inclusive-reco toys are stored below `output/unfold2D_regularization/intermediate/`. The cache filename ends in the random seed. Contents, errors, binning, toy policy, and seed are fingerprinted; a mismatched cache is ignored and regenerated.

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
# Fingerprints make cached toys invalid whenever a numerical input changes.
toy_cache_metadata = {
    'random_seed': RANDOM_SEED, 'n_toys': N_TOYS,
    'toy_distribution': TOY_DISTRIBUTION,
    'stop_on_negative': STOP_ON_NEGATIVE_TOY_BIN,
    'pt_ave_bins': PTAVE_INTERVALS, 'eta_range': ETA_UNFOLDING_RANGE,
    'toy_expectation_fingerprint': histogram_fingerprint(toy_expectation),
}
toys = (load_toy_cache(TOY_CACHE_FILE, n_toys=N_TOYS,
                       metadata=toy_cache_metadata)
        if USE_INTERMEDIATE_CACHE else None)
if toys is None:
    print(f'Generating {N_TOYS} toys for {layout.n_global_bins} bins')
    toys = make_gaussian_toys(
        toy_expectation, n_toys=N_TOYS, random_seed=RANDOM_SEED,
        layout=layout, stop_on_negative=STOP_ON_NEGATIVE_TOY_BIN)
    if USE_INTERMEDIATE_CACHE:
        write_toy_cache(TOY_CACHE_FILE, toys, toy_cache_metadata)
        print(f'Wrote toy cache: {TOY_CACHE_FILE}')
else:
    print(f'Loaded {N_TOYS} toys from cache: {TOY_CACHE_FILE}')
print(f'Total unfolding calls: {N_TOYS * MAX_ITERATIONS}')
print(f'Output: {OUTPUT_ROOT_FILE}')
# Factorized fake treatment: inclusive reco -> matched reco via purity.
signal_toys = [
    apply_purity_correction(toy, reco_purity, name=f'hRecoSignalToy_{index}')
    for index, toy in enumerate(toys)
]
toy_canvas = draw_toy_input_diagnostic(
    target_prior, toy_expectation, data_reco, toys[0],
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_toy_inputs.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID)
toy_canvas

## 7. Scan Bayesian iterations

Every purity-corrected toy is unfolded as one complete flattened distribution with the matched-event response, then divided by efficiency before comparison with the inclusive target prior. Bias is calculated and squared separately in every global bin, variance uses the N-1 denominator, and the iteration is selected from the configured aggregate flattened MSE. Optional per-pT plots only display slices of these results; they do not perform separate scans or select separate iteration counts.

For a per-bin metric $q_i$ (Bias$^2$, Variance, or MSE), **Mean metric** is $N_{bins}^{-1}\sum_i q_i$ and therefore has squared-bin-content units. This is the default selection metric: bins contribute exactly the absolute discrepancies present in the spectrum, so a high-content bin correctly has more influence. **Mean fractional metric** is $N_{valid}^{-1}\sum_i q_i/t_i^2$, where $t_i$ is the target-truth bin content and bins with negligible $t_i$ are excluded. It answers a different question: the typical fractional performance of a bin. Equal eta-bin widths require no extra width correction in either average. `REGULARIZATION_SELECTION_MODE` accepts `absolute`, `relative`, or `both`; `both` reports both minima and uses the absolute result wherever one final iteration is required.

The compact scan cache stores the mean, bias, bias-squared, variance, MSE, and summaries. When `STORE_ITERATION_DISTRIBUTIONS=True`, a dedicated seed-tagged ROOT file additionally stores the eta projections, flattened inputs and corrections, inclusive and purity-corrected toys, response matrix and diagnostics, every efficiency-corrected unfolded toy under its iteration, and all metric distributions. A compatible complete file is reused together with the scan cache, so RooUnfold is not rerun.

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
def report_progress(iterations, maximum):
    print(f'Unfolding iteration {iterations}/{maximum}')

scan_cache_metadata = {
    **toy_cache_metadata, 'max_iterations': MAX_ITERATIONS,
    'selection_mode': REGULARIZATION_SELECTION_MODE,
    'target_truth_fingerprint': histogram_fingerprint(target_prior),
    'purity_fingerprint': histogram_fingerprint(reco_purity),
    'efficiency_fingerprint': histogram_fingerprint(truth_efficiency),
    'response_fingerprint': histogram_fingerprint(response_matrix),
}
distributions_complete = (regularization_distribution_file_matches(
    ITERATION_DISTRIBUTIONS_FILE, metadata=scan_cache_metadata,
    max_iterations=MAX_ITERATIONS, n_toys=N_TOYS)
    if STORE_ITERATION_DISTRIBUTIONS else True)
scan = (load_regularization_scan_cache(
            SCAN_CACHE_FILE, max_iterations=MAX_ITERATIONS,
            metadata=scan_cache_metadata)
        if USE_INTERMEDIATE_CACHE else None)
if scan is not None and not distributions_complete:
    print('The compact scan cache exists, but the complete distribution file does not; rerunning.')
    scan = None
if scan is None:
    distribution_writer = None
    if STORE_ITERATION_DISTRIBUTIONS:
        distribution_writer = RegularizationDistributionWriter(
            ITERATION_DISTRIBUTIONS_FILE, metadata=scan_cache_metadata,
            histogram_groups={
                'eta_distributions': (
                    *truth_by_pt, *reco_by_pt, *miss_by_pt, *fake_by_pt,
                    *prior_by_pt, *data_by_pt),
                'flattened_inputs': (
                    training_truth, training_reco, training_miss, training_fake,
                    prior, target_prior, data_reco, response_matrix,
                    diagnostics.matched_truth, diagnostics.matched_measured,
                    diagnostics.effective_miss, diagnostics.effective_fake,
                    diagnostics.boundary_miss, diagnostics.boundary_fake,
                    reco_purity, truth_efficiency,
                    target_forward_fold.folded_signal, target_forward_fold.fake,
                    folded_prior, toy_expectation),
                'inclusive_reco_toys': toys,
                'purity_corrected_toys': signal_toys,
            })
    try:
        # Each callback write occurs after the efficiency correction, so the
        # stored unfolded toy represents inclusive truth used by the metrics.
        scan = scan_bayes_regularization(
            RooUnfold, matched_response_bundle, signal_toys, target_prior,
            max_iterations=MAX_ITERATIONS,
            selection_mode=REGULARIZATION_SELECTION_MODE, handle_fakes=False,
            efficiency=truth_efficiency, progress=report_progress,
            unfolded_callback=(distribution_writer.write_unfolded
                               if distribution_writer else None))
        if distribution_writer:
            distribution_writer.finalize(scan)
            print(f'Wrote complete iteration distributions: {ITERATION_DISTRIBUTIONS_FILE}')
    except Exception:
        if distribution_writer:
            distribution_writer.abort()
        raise
    if USE_INTERMEDIATE_CACHE:
        write_regularization_scan_cache(
            SCAN_CACHE_FILE, scan, scan_cache_metadata)
        print(f'Wrote scan cache: {SCAN_CACHE_FILE}')
else:
    print(f'Loaded completed scan from cache: {SCAN_CACHE_FILE}')
    if STORE_ITERATION_DISTRIBUTIONS:
        print(f'Using complete iteration distributions: {ITERATION_DISTRIBUTIONS_FILE}')
if scan.selection_mode == 'both':
    print(f'Recommended iterations from absolute MSE: {scan.absolute_selected_iterations}')
    print(f'Recommended iterations from relative MSE: {scan.relative_selected_iterations}')
    print(f'Primary iteration used below: {scan.selected_iterations} (absolute)')
else:
    print(f'Recommended iterations from {scan.selection_mode} MSE: {scan.selected_iterations}')
checked_minima = ({scan.absolute_selected_iterations, scan.relative_selected_iterations}
                   if scan.selection_mode == 'both' else {scan.selected_iterations})
if checked_minima & {1, MAX_ITERATIONS}:
    print('WARNING: at least one selected MSE minimum is at a scan boundary; '
          'extend the iteration range.')

## 8. Plot and inspect the result

In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
# Choose one stored toy and the Bayesian iterations to compare.
TOY_MODEL_NUMBER = 0          # zero-based: 0, ..., N_TOYS - 1
TOY_MODEL_ITERATIONS = (1, 4, 8)
TOY_MODEL_SEED = RANDOM_SEED  # may point to another previously stored seed

if not 0 <= TOY_MODEL_NUMBER < N_TOYS:
    raise ValueError(f'TOY_MODEL_NUMBER must be between 0 and {N_TOYS - 1}')
if not TOY_MODEL_ITERATIONS:
    raise ValueError('Select at least one Bayesian iteration')
if any(not 1 <= iteration <= MAX_ITERATIONS for iteration in TOY_MODEL_ITERATIONS):
    raise ValueError(f'Iterations must be between 1 and {MAX_ITERATIONS}')
if len(set(TOY_MODEL_ITERATIONS)) != len(TOY_MODEL_ITERATIONS):
    raise ValueError('TOY_MODEL_ITERATIONS contains duplicates')

# Read detached copies: the plots remain valid after the ROOT file is closed.
stored_file = (CACHE_DIR /
    f'{OUTPUT_BASE_TAG}_iteration_distributions_seed_{TOY_MODEL_SEED}.root')
if not stored_file.is_file():
    raise FileNotFoundError(f'Cannot open stored distributions: {stored_file}')
source_file = ROOT.TFile.Open(str(stored_file), 'READ')

def clone_stored_histogram(key, name):
    source = source_file.Get(key)
    if not source:
        raise KeyError(f'Missing {key} in {stored_file}')
    histogram = source.Clone(name)
    histogram.SetDirectory(0)
    return histogram

try:
    stored_truth = clone_stored_histogram(
        'flattened_inputs/hMixedPriorDataYieldTruth', 'hStoredTargetTruth')
    stored_templates = [clone_stored_histogram(
        f'eta_distributions/hTrainingTruth_pt{pt_index}_restricted',
        f'hStoredEtaTemplate_pt{pt_index}')
        for pt_index in range(len(PTAVE_INTERVALS))]
    stored_unfolded = {iteration: clone_stored_histogram(
        f'iteration_{iteration:03d}/unfolded_toys/'
        f'hUnfoldedToy_{TOY_MODEL_NUMBER:06d}',
        f'hStoredUnfoldedToy{TOY_MODEL_NUMBER}_iter{iteration}')
        for iteration in TOY_MODEL_ITERATIONS}
finally:
    source_file.Close()

stored_layout = FlattenedBinning(PTAVE_INTERVALS, stored_templates[0].GetNbinsX())
n_columns = math.ceil(math.sqrt(stored_layout.n_pt_bins))
n_rows = math.ceil(stored_layout.n_pt_bins / n_columns)
overlay_canvas = ROOT.TCanvas('canvas_stored_toy_overlay', '', 450*n_columns, 350*n_rows)
ratio_canvas = ROOT.TCanvas('canvas_stored_toy_ratio', '', 450*n_columns, 350*n_rows)
overlay_canvas.Divide(n_columns, n_rows)
ratio_canvas.Divide(n_columns, n_rows)
colors = (ROOT.kRed + 1, ROOT.kBlue + 1, ROOT.kGreen + 2,
          ROOT.kMagenta + 1, ROOT.kOrange + 7, ROOT.kCyan + 2)
retained_overlay, retained_ratio = [], []

for pt_index, (pt_low, pt_high) in enumerate(PTAVE_INTERVALS):
    truth_eta = extract_eta_block(
        stored_truth, stored_templates[pt_index], stored_layout, pt_index,
        name=f'hStoredTruthEta_pt{pt_index}')
    unfolded_eta = {iteration: extract_eta_block(
        histogram, stored_templates[pt_index], stored_layout, pt_index,
        name=f'hStoredUnfoldedEta_toy{TOY_MODEL_NUMBER}_iter{iteration}_pt{pt_index}')
        for iteration, histogram in stored_unfolded.items()}

    overlay_canvas.cd(pt_index + 1)
    ROOT.gPad.SetGrid(DRAW_GRID, DRAW_GRID)
    truth_eta.SetTitle(f'{pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV;#eta_{{CM}};Entries')
    truth_eta.SetLineColor(ROOT.kBlack)
    truth_eta.SetLineWidth(3)
    truth_eta.SetMarkerStyle(20)
    maximum = max([truth_eta.GetMaximum(),
                   *(histogram.GetMaximum() for histogram in unfolded_eta.values())])
    truth_eta.SetMaximum(1.25 * maximum if maximum > 0 else 1.0)
    truth_eta.Draw('HIST')
    overlay_legend = ROOT.TLegend(0.54, 0.66, 0.89, 0.89)
    overlay_legend.SetBorderSize(0)
    overlay_legend.AddEntry(truth_eta, 'Target truth', 'l')
    for color_index, (iteration, histogram) in enumerate(unfolded_eta.items()):
        histogram.SetLineColor(colors[color_index % len(colors)])
        histogram.SetLineWidth(2)
        histogram.Draw('HIST SAME')
        overlay_legend.AddEntry(histogram, f'Unfolded, {iteration} iter.', 'l')
    overlay_legend.Draw()
    retained_overlay.extend((truth_eta, overlay_legend, *unfolded_eta.values()))

    ratio_canvas.cd(pt_index + 1)
    ROOT.gPad.SetGrid(DRAW_GRID, DRAW_GRID)
    ratios = []
    for color_index, (iteration, histogram) in enumerate(unfolded_eta.items()):
        ratio = histogram.Clone(
            f'hStoredUnfoldedToTruth_toy{TOY_MODEL_NUMBER}_iter{iteration}_pt{pt_index}')
        ratio.SetDirectory(0)
        ratio.Divide(truth_eta)
        ratio.SetTitle(f'{pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV;#eta_{{CM}};Unfolded / target truth')
        ratio.SetLineColor(colors[color_index % len(colors)])
        ratio.SetLineWidth(2)
        ratio.SetMinimum(0.5)
        ratio.SetMaximum(1.5)
        ratio.Draw('HIST' if color_index == 0 else 'HIST SAME')
        ratios.append((iteration, ratio))
    unity = ROOT.TLine(truth_eta.GetXaxis().GetXmin(), 1.0,
                       truth_eta.GetXaxis().GetXmax(), 1.0)
    unity.SetLineStyle(2)
    unity.Draw()
    ratio_legend = ROOT.TLegend(0.57, 0.70, 0.89, 0.89)
    ratio_legend.SetBorderSize(0)
    for iteration, ratio in ratios:
        ratio_legend.AddEntry(ratio, f'{iteration} iter.', 'l')
    ratio_legend.Draw()
    retained_ratio.extend((*[ratio for _, ratio in ratios], unity, ratio_legend))

overlay_canvas._stored_objects = retained_overlay
ratio_canvas._stored_objects = retained_ratio
overlay_canvas.Update()
ratio_canvas.Update()
inspection_tag = f'{OUTPUT_BASE_TAG}_toy{TOY_MODEL_NUMBER}_seed_{TOY_MODEL_SEED}'
overlay_canvas.Print(str(OUTPUT_DIR / f'{inspection_tag}_unfolded_overlay.pdf'))
ratio_canvas.Print(str(OUTPUT_DIR / f'{inspection_tag}_unfolded_to_truth.pdf'))
print(f'Read stored toy distributions from: {stored_file}')
overlay_canvas, ratio_canvas

In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
# Select inclusive-reco toys: these are fluctuations of the forward-folded target.
SMEARED_TOY_NUMBERS = (0, 1, 2)  # zero-based: 0, ..., N_TOYS - 1
SMEARED_TOY_SEED = RANDOM_SEED
SMEARED_RATIO_RANGE = (0.5, 1.5)

if not SMEARED_TOY_NUMBERS:
    raise ValueError('Select at least one smeared toy')
if len(set(SMEARED_TOY_NUMBERS)) != len(SMEARED_TOY_NUMBERS):
    raise ValueError('SMEARED_TOY_NUMBERS contains duplicates')
if any(not 0 <= toy_number < N_TOYS for toy_number in SMEARED_TOY_NUMBERS):
    raise ValueError(f'Toy numbers must be between 0 and {N_TOYS - 1}')
if not SMEARED_RATIO_RANGE[0] < SMEARED_RATIO_RANGE[1]:
    raise ValueError('SMEARED_RATIO_RANGE must be increasing')

smeared_file = (CACHE_DIR /
    f'{OUTPUT_BASE_TAG}_iteration_distributions_seed_{SMEARED_TOY_SEED}.root')
if not smeared_file.is_file():
    raise FileNotFoundError(f'Cannot open stored distributions: {smeared_file}')
source_file = ROOT.TFile.Open(str(smeared_file), 'READ')

def clone_smeared_histogram(key, name):
    source = source_file.Get(key)
    if not source:
        raise KeyError(f'Missing {key} in {smeared_file}')
    histogram = source.Clone(name)
    histogram.SetDirectory(0)
    return histogram

try:
    stored_expectation = clone_smeared_histogram(
        'flattened_inputs/hRecoToyExpectation', 'hStoredRecoToyExpectation')
    stored_reco_templates = [clone_smeared_histogram(
        f'eta_distributions/hTrainingReco_pt{pt_index}_restricted',
        f'hStoredRecoTemplate_pt{pt_index}')
        for pt_index in range(len(PTAVE_INTERVALS))]
    stored_smeared_toys = {toy_number: clone_smeared_histogram(
        f'inclusive_reco_toys/hRecoToy_{toy_number}',
        f'hStoredSmearedToy{toy_number}')
        for toy_number in SMEARED_TOY_NUMBERS}
finally:
    source_file.Close()

stored_reco_layout = FlattenedBinning(
    PTAVE_INTERVALS, stored_reco_templates[0].GetNbinsX())
smeared_canvases, smeared_ratios = [], []
for pt_index, (pt_low, pt_high) in enumerate(PTAVE_INTERVALS):
    expectation_eta = extract_eta_block(
        stored_expectation, stored_reco_templates[pt_index], stored_reco_layout,
        pt_index, name=f'hStoredRecoExpectationEta_pt{pt_index}')
    displayed = {'Forward-folded expectation': expectation_eta}
    for toy_number, toy in stored_smeared_toys.items():
        displayed[f'Smeared toy {toy_number}'] = extract_eta_block(
            toy, stored_reco_templates[pt_index], stored_reco_layout, pt_index,
            name=f'hStoredSmearedToy{toy_number}Eta_pt{pt_index}')
    canvas, ratios = draw_closure(
        displayed, nominal='Forward-folded expectation',
        title=f'{pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV',
        x_title='#eta_{CM}', y_title='Reco entries',
        ratio_range=SMEARED_RATIO_RANGE,
        draw_nominal_ratio=False, grid=DRAW_GRID,
        annotations=(f'Seed {SMEARED_TOY_SEED}',),
        output=OUTPUT_DIR /
            f'{OUTPUT_BASE_TAG}_smeared_toys_seed_{SMEARED_TOY_SEED}_'
            f'pt_{pt_low:g}_{pt_high:g}.pdf',
        save_png=SAVE_PNG,
        canvas_name=f'canvas_smeared_toys_pt{pt_index}')
    smeared_canvases.append(canvas)
    smeared_ratios.extend(ratios.values())
print(f'Read stored smeared toys from: {smeared_file}')
smeared_canvases

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
absolute_canvas, absolute_metric_plots = draw_regularization_metrics(
    scan, mode='absolute',
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_absolute_metrics.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID,
    log_y=LOG_REGULARIZATION_METRICS_Y)
relative_canvas, relative_metric_plots = draw_regularization_metrics(
    scan, mode='relative',
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_relative_metrics.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID,
    log_y=LOG_REGULARIZATION_METRICS_Y)
per_pt_metric_canvases = []
per_pt_metric_histograms = []
if PLOT_PER_PT_METRICS:
    canvases, histograms = draw_regularization_metrics_by_pt(
        scan, layout, output_dir=OUTPUT_DIR, output_tag=OUTPUT_TAG,
        mode='absolute', save_png=SAVE_PNG, grid=DRAW_GRID,
        log_y=LOG_REGULARIZATION_METRICS_Y)
    per_pt_metric_canvases.extend(canvases)
    per_pt_metric_histograms.extend(histograms)
    canvases, histograms = draw_regularization_metrics_by_pt(
        scan, layout, output_dir=OUTPUT_DIR, output_tag=OUTPUT_TAG,
        mode='relative', save_png=SAVE_PNG, grid=DRAW_GRID,
        log_y=LOG_REGULARIZATION_METRICS_Y)
    per_pt_metric_canvases.extend(canvases)
    per_pt_metric_histograms.extend(histograms)
# The tuple is ordered by iteration and therefore uses iteration - 1 indexing.
selected_metrics = scan.metrics[scan.selected_iterations - 1]
truth_closure_canvas, truth_closure_ratios = draw_closure(
    {'Target truth': target_prior, 'Mean unfolded toys': selected_metrics.mean},
    nominal='Target truth', title='', x_title='global truth bin',
    y_title='Entries', ratio_range=(0.5, 1.5), draw_nominal_ratio=False,
    annotations=(f'{scan.selected_iterations} Bayesian iterations',),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_selected_truth_closure.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID,
    canvas_name='canvas_selected_truth_closure')
mean_to_prior = truth_closure_ratios['Mean unfolded toys']
mean_to_prior.SetName('hSelectedMeanUnfoldedToTargetTruth')
selected_mean_forward_fold = forward_fold_truth(
    inclusive_response_bundle, selected_metrics.mean, diagnostics=diagnostics,
    add_fakes=True, fake_normalization='matched_fraction',
    name='hSelectedMeanForwardFolded')
reco_closure_canvas, reco_closure_ratios = draw_closure(
    {'Toy expectation': toy_expectation,
     'Forward-folded mean': selected_mean_forward_fold.histogram},
    nominal='Toy expectation', title='', x_title='global measured bin',
    y_title='Entries', ratio_range=(0.5, 1.5), draw_nominal_ratio=False,
    annotations=(f'{scan.selected_iterations} Bayesian iterations',),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_selected_refolded_closure.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID,
    canvas_name='canvas_selected_refolded_closure')
refolded_to_expectation = reco_closure_ratios['Forward-folded mean']
refolded_to_expectation.SetName('hSelectedRefoldedMeanToToyExpectation')
display(absolute_canvas)
display(relative_canvas)
display(truth_closure_canvas)
display(reco_closure_canvas)

## 9. Save results and print the final summary

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
selected_signal_expectation = apply_purity_correction(
    toy_expectation, reco_purity, name='hSelectedRecoSignalExpectation')
selected_matched_unfolding = unfold_bayes(
    RooUnfold, matched_response_bundle, selected_signal_expectation,
    iterations=scan.selected_iterations, handle_fakes=False,
    name='hSelectedIterationUnfoldedMatchedExpectation')
selected_unfolded, selected_covariance = apply_efficiency_correction(
    selected_matched_unfolding.histogram, selected_matched_unfolding.covariance,
    truth_efficiency, name='hSelectedIterationUnfoldedExpectation')
metric_histograms = tuple(
    histogram
    for result in scan.metrics
    for histogram in (result.mean, result.bias, result.bias_squared, result.variance, result.mse)
)
valid_mask = target_prior.Clone('hRelativeMetricValidBins')
valid_mask.Reset('ICES')
for bin_index in selected_metrics.valid_relative_bins:
    valid_mask.SetBinContent(bin_index, 1.0)

write_unfolding_output(
    OUTPUT_ROOT_FILE,
    histograms=(
        prior, target_prior, target_forward_fold.folded_signal,
        target_forward_fold.fake, folded_prior, toy_expectation,
        selected_signal_expectation, reco_purity, truth_efficiency,
        data_reco, response_matrix, valid_mask,
        scan.absolute_summary, scan.relative_summary,
        selected_matched_unfolding.histogram, selected_unfolded,
        mean_to_prior, selected_mean_forward_fold.folded_signal,
        selected_mean_forward_fold.fake, selected_mean_forward_fold.histogram,
        refolded_to_expectation,
        *metric_histograms, *absolute_metric_plots, *relative_metric_plots,
        *per_pt_metric_histograms,
    ),
    covariance=selected_covariance,
    covariance_name='hSelectedIterationCovariance',
    response=matched_response_bundle.response,
    metadata={
        'mc_file': str(mc_file), 'response_keys': keys.__dict__,
        'generator': GENERATOR, 'direction': DIRECTION,
        'eta_cut': ETA_CUT, 'eta_unfolding_range': ETA_UNFOLDING_RANGE,
        'pt_ave_bins': PTAVE_INTERVALS,
        'ptave_bin_set': PTAVE_BIN_SET, 'trigger_by_interval': {
            f'{low:g}_{high:g}': TRIGGER_BY_INTERVAL[(low, high)]
            for low, high in PTAVE_INTERVALS},
        'n_toys': N_TOYS, 'random_seed': RANDOM_SEED,
        'regularization_selection_mode': REGULARIZATION_SELECTION_MODE,
        'absolute_recommended_iterations': scan.absolute_selected_iterations,
        'relative_recommended_iterations': scan.relative_selected_iterations,
        'use_intermediate_cache': USE_INTERMEDIATE_CACHE,
        'log_regularization_metrics_y': LOG_REGULARIZATION_METRICS_Y,
        'store_iteration_distributions': STORE_ITERATION_DISTRIBUTIONS,
        'toy_cache_file': str(TOY_CACHE_FILE),
        'scan_cache_file': str(SCAN_CACHE_FILE),
        'iteration_distributions_file': str(ITERATION_DISTRIBUTIONS_FILE),
        'max_iterations': MAX_ITERATIONS, 'response_scale': RESPONSE_SCALE,
        'unfolding_method': 'factorized_purity_migration_efficiency',
        'forward_fold_fake_normalization': 'binwise_fake_to_matched_reco',
        'apply_to_truth_includes_fakes': target_forward_fold.apply_to_truth_includes_fakes,
        'block_scale_factors': block_scale_factors,
        'yield_match_iterations': yield_match.iterations,
        'yield_match_maximum_relative_residual': yield_match.maximum_relative_residual,
        'yield_match_tolerance': YIELD_MATCH_TOLERANCE,
        'yield_match_relaxation': YIELD_MATCH_RELAXATION,
        'negative_bin_policy': 'stop' if STOP_ON_NEGATIVE_TOY_BIN else 'allow',
        'excluded_relative_bins': [
            index for index in range(1, layout.n_global_bins + 1)
            if index not in selected_metrics.valid_relative_bins],
        'recommended_iterations': scan.selected_iterations,
    },
)
print(f'Wrote {OUTPUT_ROOT_FILE}')
print(f'Primary Bayesian iterations: {scan.selected_iterations}')
print(f'Absolute / relative minima: {scan.absolute_selected_iterations} / '
      f'{scan.relative_selected_iterations}')